# Travel Reimbursement Approval Agent — `ajaybeniwal.ipynb`

## README / Setup

**Assignment:** AI Developer Candidate Assignment — Travel Reimbursement Approval Agent (Appendix A policy, Appendix B claims).

### Install dependencies
```bash
pip install openai pydantic ipywidgets jupyter nbformat python-dotenv
```

### Prerequisites
- Python 3.10+ (tested with 3.14)
- `python-dotenv` is **only required** when loading credentials from a local `.env` file (not needed if `OPENAI_API_KEY` is already exported)

### Environment variables
| Variable | Required | Purpose |
|----------|----------|---------|
| `OPENAI_API_KEY` | **Yes** for live evaluation | OpenAI API authentication |
| `OPENAI_MODEL` | No (default `gpt-4o-mini`) | Chat Completions model with tool calling |

**Model choice:** `gpt-4o-mini` supports [Chat Completions tool calling](https://platform.openai.com/docs/guides/function-calling) and structured JSON recommendations at modest cost.

### Credentials (local only — never commit)
```bash
export OPENAI_API_KEY="sk-..."
export OPENAI_MODEL="gpt-4o-mini"   # optional
```

Or create a `.env` file next to this notebook (listed in `.gitignore`):
```
OPENAI_API_KEY=sk-...
OPENAI_MODEL=gpt-4o-mini
```

### Run
1. Start Jupyter / VS Code notebook kernel.
2. **Run All** — evaluates all five claims once (no mid-run prompts).
3. Inspect **Dashboard**, **Sample outputs**, and the **final JSON** cell.

Without `OPENAI_API_KEY`, a **ConfigurationError** stops before LLM evaluation (no fabricated demo results).

### Receipt evidence assumption
For this mock dataset, a line item `receipt_attached: true` is treated as sufficient evidence of a receipt. No receipt files or itemization are supplied; the agent does **not** perform OCR or file inspection.

In [1]:
# Load local .env predictably (never commit secrets)
import os
from pathlib import Path


class ConfigurationError(RuntimeError):
    pass


NOTEBOOK_DIR = Path.cwd().resolve()


def load_notebook_env() -> None:
    if os.environ.get("OPENAI_API_KEY"):
        return
    env_path = NOTEBOOK_DIR / ".env"
    if not env_path.is_file():
        raise ConfigurationError(
            "OPENAI_API_KEY is not set and no .env file exists in the notebook directory."
        )
    try:
        from dotenv import load_dotenv
    except ImportError as exc:
        raise ConfigurationError(
            "python-dotenv is required to load a local .env file. Install: pip install python-dotenv"
        ) from exc
    load_dotenv(env_path, override=False)


load_notebook_env()
if not os.environ.get("OPENAI_API_KEY"):
    raise ConfigurationError(
        "OPENAI_API_KEY is not set. Export it or add it to .env in this directory."
    )

## Policy (Appendix A) & derived limits

Rules and limits are embedded in code below. Every rule is citeable via `POL-*` ids.

**Approval matrix (after per-diem deductions):**
| Tier | Rule | Reimbursable total |
|------|------|--------------------|
| Auto | POL-APR-01 | ≤ $500 |
| Manager | POL-APR-02 | > $500 and ≤ $2,000 |
| Director / Manual | POL-APR-03 | > $2,000 |

**Timeliness (documented assumption):** Expenses are assumed to occur within the trip interval. Submission ≤ trip start + 30 days ⇒ timely for any date in the window; submission > trip end + 30 ⇒ late; otherwise uncertain → Manual Review.

In [2]:
"""
Travel Reimbursement Approval Agent — core logic (embedded in ajaybeniwal.ipynb).
Development module for local verification; notebook is self-contained.
"""

from __future__ import annotations

import hashlib
import json
import os
import re
import textwrap
from dataclasses import dataclass, field
from datetime import date, timedelta
from decimal import Decimal, ROUND_HALF_UP
from enum import Enum
from typing import Any, Callable, Literal

from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError, field_validator

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

DEFAULT_OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
MAX_AGENT_ITERATIONS = 10
MAX_TOOL_RESULT_CHARS = 12_000
OPENAI_TIMEOUT_SECONDS = 120.0
MAX_JSON_DECODE_RETRIES = 1

DECISION_VALUES = frozenset({"APPROVE", "PARTIAL_APPROVE", "REJECT", "MANUAL_REVIEW"})

VALID_POLICY_IDS = frozenset(
    {
        "POL-CAT-01",
        "POL-CAT-02",
        "POL-PD-01",
        "POL-PD-02",
        "POL-PD-03",
        "POL-AIR-01",
        "POL-RCT-01",
        "POL-RCT-02",
        "POL-APR-01",
        "POL-APR-02",
        "POL-APR-03",
        "POL-TIME-01",
    }
)

RECEIPT_ATTACHMENT_COUNTS_AS_EVIDENCE = True

# ---------------------------------------------------------------------------
# Policy (Appendix A) — embedded
# ---------------------------------------------------------------------------

POLICY_RULES: dict[str, str] = {
    "POL-CAT-01": "Eligible categories: airfare (economy), lodging, meals, ground transport, conference fees.",
    "POL-CAT-02": "Ineligible: alcohol/minibar, spa/gym/entertainment, personal expenses, fines — never reimbursable.",
    "POL-PD-01": "Meals maximum $75 per day; excess deducted.",
    "POL-PD-02": "Lodging maximum $200 per night; excess deducted.",
    "POL-PD-03": "Ground transport maximum $50 per day; excess deducted.",
    "POL-AIR-01": "Only economy airfare reimbursable; business/first routed to Manual Review (pre-approval may exist).",
    "POL-RCT-01": "Line items >$25 need itemized receipt; airfare and lodging always require receipt.",
    "POL-RCT-02": "Missing required receipt → Manual Review (not silent rejection).",
    "POL-APR-01": "Auto-approve tier: reimbursable total after per-diem deductions ≤ $500 when fully compliant.",
    "POL-APR-02": "Manager tier: total > $500 and ≤ $2,000 when fully compliant.",
    "POL-APR-03": "Total > $2,000 after per-diem deductions → Manual Review (director approval).",
    "POL-TIME-01": "Claims within 30 days of expense date; late → Manual Review.",
}

LIMITS = {
    "meals_per_day": Decimal("75.00"),
    "lodging_per_night": Decimal("200.00"),
    "ground_per_day": Decimal("50.00"),
    "receipt_threshold": Decimal("25.00"),
    "auto_approve_max": Decimal("500.00"),
    "manager_max": Decimal("2000.00"),
}

INELIGIBLE_CATEGORIES = frozenset(
    {"spa", "minibar", "alcohol", "gym", "entertainment", "fines", "personal"}
)

ELIGIBLE_CATEGORIES = frozenset(
    {"airfare", "lodging", "meals", "ground_transport", "conference_fees"}
)

ALWAYS_RECEIPT_CATEGORIES = frozenset({"airfare", "lodging"})

CLAIM_SCOPED_TOOLS = frozenset(
    {
        "check_category_eligibility",
        "check_receipts",
        "calculate_per_diem",
        "check_timeliness",
        "check_airfare_exception",
        "check_approval_tier",
    }
)

# ---------------------------------------------------------------------------
# Claims (Appendix B) — faithful transcription
# ---------------------------------------------------------------------------

CLAIMS: list[dict[str, Any]] = [
    {
        "claim_id": "CLM-001",
        "employee": "A. Rivera",
        "trip_start": "2026-06-10",
        "trip_end": "2026-06-12",
        "submitted_on": "2026-06-20",
        "line_items": [
            {
                "category": "airfare",
                "description": "Round-trip economy airfare",
                "amount": "420.00",
                "receipt_attached": True,
            },
            {
                "category": "lodging",
                "description": "Hotel, 2 nights @ $180",
                "amount": "360.00",
                "receipt_attached": True,
            },
            {
                "category": "meals",
                "description": "Meals, 3 days @ ~$60/day",
                "amount": "180.00",
                "receipt_attached": True,
            },
            {
                "category": "conference_fees",
                "description": "Conference registration",
                "amount": "150.00",
                "receipt_attached": True,
            },
        ],
    },
    {
        "claim_id": "CLM-002",
        "employee": "B. Osei",
        "trip_start": "2026-06-14",
        "trip_end": "2026-06-15",
        "submitted_on": "2026-06-25",
        "line_items": [
            {
                "category": "spa",
                "description": "Hotel spa package",
                "amount": "300.00",
                "receipt_attached": True,
            },
            {
                "category": "minibar",
                "description": "In-room minibar",
                "amount": "80.00",
                "receipt_attached": True,
            },
        ],
    },
    {
        "claim_id": "CLM-003",
        "employee": "C. Nakamura",
        "trip_start": "2026-06-08",
        "trip_end": "2026-06-10",
        "submitted_on": "2026-06-22",
        "line_items": [
            {
                "category": "airfare",
                "description": "Round-trip economy airfare",
                "amount": "300.00",
                "receipt_attached": True,
            },
            {
                "category": "lodging",
                "description": "Hotel, 2 nights @ $250",
                "amount": "500.00",
                "receipt_attached": True,
            },
            {
                "category": "meals",
                "description": "Meals, 2 days @ $70/day",
                "amount": "140.00",
                "receipt_attached": True,
            },
        ],
    },
    {
        "claim_id": "CLM-004",
        "employee": "D. Fischer",
        "trip_start": "2026-06-16",
        "trip_end": "2026-06-18",
        "submitted_on": "2026-06-28",
        "line_items": [
            {
                "category": "airfare",
                "description": "Business-class international airfare",
                "amount": "2400.00",
                "receipt_attached": True,
            },
            {
                "category": "lodging",
                "description": "Hotel, 3 nights",
                "amount": "600.00",
                "receipt_attached": False,
            },
        ],
        "notes": [
            "Lodging description states 3 nights while trip 2026-06-16→2026-06-18 spans two calendar overnights if checkout on the 18th."
        ],
    },
    {
        "claim_id": "CLM-005",
        "employee": "E. Haddad",
        "trip_start": "2026-06-11",
        "trip_end": "2026-06-11",
        "submitted_on": "2026-06-24",
        "line_items": [
            {
                "category": "meals",
                "description": "Client dinner for 4 (business development)",
                "amount": "220.00",
                "receipt_attached": False,
            },
        ],
    },
]

CLAIM_BY_ID: dict[str, dict[str, Any]] = {c["claim_id"]: c for c in CLAIMS}


def money(value: Decimal | str | float) -> Decimal:
    if isinstance(value, Decimal):
        d = value
    else:
        d = Decimal(str(value))
    return d.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)


def money_float(d: Decimal) -> float:
    return float(money(d))


def parse_date(s: str) -> date:
    y, m, d = map(int, s.split("-"))
    return date(y, m, d)


def claim_total(claim: dict[str, Any]) -> Decimal:
    return sum(money(li["amount"]) for li in claim["line_items"])


def trip_calendar_nights(trip_start: date, trip_end: date) -> int:
    return max(0, (trip_end - trip_start).days)


# ---------------------------------------------------------------------------
# Pydantic models
# ---------------------------------------------------------------------------


class LineItemModel(BaseModel):
    category: str
    description: str
    amount: str
    receipt_attached: bool


class ClaimModel(BaseModel):
    claim_id: str
    employee: str
    trip_start: str
    trip_end: str
    submitted_on: str
    line_items: list[LineItemModel]
    notes: list[str] = Field(default_factory=list)


class LLMRecommendation(BaseModel):
    claim_id: str
    decision: Literal["APPROVE", "PARTIAL_APPROVE", "REJECT", "MANUAL_REVIEW"]
    confidence: float = Field(ge=0.0, le=1.0)
    explanation: str = Field(min_length=1, max_length=8000)
    policy_refs: list[str] = Field(default_factory=list)
    missing_docs: list[str] = Field(default_factory=list)

    @field_validator("policy_refs")
    @classmethod
    def policy_refs_valid(cls, v: list[str]) -> list[str]:
        unknown = [p for p in v if p not in VALID_POLICY_IDS]
        if unknown:
            raise ValueError(f"Unknown policy refs: {unknown}")
        return v


class ClaimResult(BaseModel):
    claim_id: str
    decision: Literal["APPROVE", "PARTIAL_APPROVE", "REJECT", "MANUAL_REVIEW"]
    approved_amount: float
    deducted_amount: float
    missing_docs: list[str]
    policy_refs: list[str]
    confidence: float
    explanation: str
    tools_used: list[str]
    pending_amount: float = 0.0
    audit_trail: list[dict[str, Any]] = Field(default_factory=list)
    safety_overrides: list[str] = Field(default_factory=list)
    technical_failure: bool = False

    @field_validator("policy_refs")
    @classmethod
    def policy_refs_valid(cls, v: list[str]) -> list[str]:
        bad = [p for p in v if p not in VALID_POLICY_IDS]
        if bad:
            raise ValueError(f"Invalid policy refs: {bad}")
        return v


class EvaluationState(BaseModel):
    """Ephemeral per-claim evidence collected via tools."""

    claim_id: str
    receipt_check: dict[str, Any] | None = None
    per_diem: dict[str, Any] | None = None
    timeliness: dict[str, Any] | None = None
    approval_tier: dict[str, Any] | None = None
    airfare_exception: dict[str, Any] | None = None
    category_eligibility: dict[str, Any] | None = None
    policy_lookups: list[str] = Field(default_factory=list)


# ---------------------------------------------------------------------------
# Parsing aggregated expense descriptions
# ---------------------------------------------------------------------------

NIGHTS_RE = re.compile(r"(?P<n>\d+)\s*nights?", re.I)
DAYS_RE = re.compile(r"(?P<n>\d+)\s*days?", re.I)
RATE_RE = re.compile(r"@\s*~?\$?(?P<rate>[\d.]+)\s*/\s*(?:night|day)", re.I)


@dataclass
class AggregationNote:
    line_index: int
    kind: str
    message: str


def parse_meals_lodging_line(item: dict[str, Any], idx: int) -> tuple[dict[str, Any], list[AggregationNote]]:
    notes: list[AggregationNote] = []
    desc = item["description"]
    amount = money(item["amount"])
    cat = item["category"]
    meta: dict[str, Any] = {"line_index": idx, "category": cat, "amount": money_float(amount)}

    rate_m = RATE_RE.search(desc)
    if cat == "lodging":
        nights_m = NIGHTS_RE.search(desc)
        if nights_m and rate_m:
            n = int(nights_m.group("n"))
            rate = money(rate_m.group("rate"))
            meta.update({"nights": n, "rate_per_night": money_float(rate), "parsed_from": "description"})
            cap = LIMITS["lodging_per_night"] * n
            approved = min(amount, cap)
            if rate > LIMITS["lodging_per_night"]:
                notes.append(
                    AggregationNote(idx, "lodging", f"Stated ${rate}/night exceeds $200 cap for {n} night(s).")
                )
        elif nights_m and not rate_m:
            n = int(nights_m.group("n"))
            implied = (amount / n) if n else amount
            meta.update(
                {
                    "nights": n,
                    "implied_rate_per_night": money_float(implied),
                    "parsed_from": "amount_div_nights",
                }
            )
            cap = LIMITS["lodging_per_night"] * n
            approved = min(amount, cap)
            if implied > LIMITS["lodging_per_night"]:
                notes.append(
                    AggregationNote(
                        idx,
                        "lodging",
                        f"Implied ${money(implied)}/night from total/nights exceeds cap.",
                    )
                )
        else:
            meta["parsed_from"] = "none"
            approved = amount
            notes.append(AggregationNote(idx, "lodging", "Could not parse nights/rate; per-diem not applied."))
        meta["approved_line"] = money_float(approved)
        meta["deducted_line"] = money_float(amount - approved)
        return meta, notes

    if cat == "meals":
        days_m = DAYS_RE.search(desc)
        approx = "~" in desc
        if days_m and rate_m:
            n = int(days_m.group("n"))
            rate = money(rate_m.group("rate"))
            meta.update(
                {
                    "days": n,
                    "rate_per_day": money_float(rate),
                    "approximate_rate": approx,
                    "parsed_from": "description",
                }
            )
            if approx:
                notes.append(
                    AggregationNote(
                        idx,
                        "meals",
                        "Description uses ~$60/day approximation; daily compliance assumed at stated rate only.",
                    )
                )
            cap = LIMITS["meals_per_day"] * n
            if rate > LIMITS["meals_per_day"]:
                notes.append(AggregationNote(idx, "meals", f"Stated ${rate}/day exceeds $75/day for {n} day(s)."))
            approved = min(amount, cap)
        elif days_m and not rate_m:
            n = int(days_m.group("n"))
            implied = amount / n if n else amount
            meta.update({"days": n, "implied_rate_per_day": money_float(implied), "parsed_from": "amount_div_days"})
            cap = LIMITS["meals_per_day"] * n
            approved = min(amount, cap)
            if implied > LIMITS["meals_per_day"]:
                notes.append(AggregationNote(idx, "meals", f"Implied ${money(implied)}/day exceeds cap."))
        else:
            # Single aggregated meal without day count (CLM-005)
            meta.update({"days": 1, "parsed_from": "single_day_assumption", "single_line_meal": True})
            cap = LIMITS["meals_per_day"]
            approved = min(amount, cap)
            notes.append(
                AggregationNote(
                    idx,
                    "meals",
                    "Single aggregated meal line; daily cap applied to line total for one trip day (provisional if under review).",
                )
            )
        meta["approved_line"] = money_float(approved)
        meta["deducted_line"] = money_float(amount - approved)
        return meta, notes

    meta.update({"approved_line": money_float(amount), "deducted_line": 0.0})
    return meta, notes


def parse_ground_transport_line(
    item: dict[str, Any], idx: int, trip_start: date, trip_end: date
) -> tuple[dict[str, Any], list[AggregationNote]]:
    notes: list[AggregationNote] = []
    desc = item["description"]
    amount = money(item["amount"])
    meta: dict[str, Any] = {"line_index": idx, "category": item["category"], "amount": money_float(amount)}
    days_m = DAYS_RE.search(desc)
    rate_m = RATE_RE.search(desc)
    trip_days = max(1, (trip_end - trip_start).days + 1)
    if days_m and rate_m:
        n = int(days_m.group("n"))
        rate = money(rate_m.group("rate"))
        meta.update({"days": n, "rate_per_day": money_float(rate), "parsed_from": "description"})
        if rate > LIMITS["ground_per_day"]:
            notes.append(AggregationNote(idx, "ground_transport", f"Stated ${rate}/day exceeds $50/day (POL-PD-03)."))
        cap = LIMITS["ground_per_day"] * n
        approved = min(amount, cap)
    elif days_m:
        n = int(days_m.group("n"))
        implied = amount / n if n else amount
        meta.update({"days": n, "implied_rate_per_day": money_float(implied), "parsed_from": "amount_div_days"})
        if implied > LIMITS["ground_per_day"]:
            notes.append(AggregationNote(idx, "ground_transport", "Implied rate exceeds $50/day (POL-PD-03)."))
        cap = LIMITS["ground_per_day"] * n
        approved = min(amount, cap)
    elif rate_m and not days_m:
        meta.update({"parsed_from": "ambiguous_days", "requires_manual_review": True})
        notes.append(
            AggregationNote(
                idx,
                "ground_transport",
                "Rate present without day count; daily allocation ambiguous—manual review required.",
            )
        )
        approved = Decimal("0")
        cap = Decimal("0")
    else:
        if trip_days > 1:
            meta.update({"parsed_from": "ambiguous_allocation", "requires_manual_review": True})
            notes.append(
                AggregationNote(
                    idx,
                    "ground_transport",
                    "Multi-day trip without daily allocation evidence; manual review required (POL-PD-03).",
                )
            )
            approved = Decimal("0")
        else:
            meta.update({"days": 1, "parsed_from": "single_trip_day_deferred", "pending_daily_aggregate": True})
            approved = amount
    meta["approved_line"] = money_float(approved)
    meta["deducted_line"] = money_float(amount - approved)
    meta["policy_ref"] = "POL-PD-03"
    return meta, notes


def classify_airfare_cabin(description: str) -> Literal["economy", "business", "unknown"]:
    d = description.lower()
    if "business" in d or "first-class" in d or "first class" in d:
        return "business"
    if "economy" in d:
        return "economy"
    return "unknown"


def is_economy_airfare(description: str) -> bool:
    """Deprecated helper; use classify_airfare_cabin."""
    return classify_airfare_cabin(description) == "economy"


# ---------------------------------------------------------------------------
# Deterministic policy tools (canonical data only)
# ---------------------------------------------------------------------------


def tool_lookup_policy(rule_ids: list[str]) -> dict[str, Any]:
    rule_ids = list(dict.fromkeys(rule_ids))
    invalid = [r for r in rule_ids if r not in VALID_POLICY_IDS]
    if invalid:
        return {"error": f"Unknown rule ids: {invalid}", "rules": {}}
    return {"rules": {rid: POLICY_RULES[rid] for rid in rule_ids}}


def tool_category_eligibility(claim_id: str) -> dict[str, Any]:
    claim = CLAIM_BY_ID.get(claim_id)
    if not claim:
        return {"error": f"Unknown claim_id {claim_id}"}
    lines = []
    all_ineligible = True
    any_eligible = False
    any_unknown = False
    for i, li in enumerate(claim["line_items"]):
        cat = li["category"].lower()
        if cat in INELIGIBLE_CATEGORIES:
            status = "ineligible"
            all_ineligible = all_ineligible and True
        elif cat in ELIGIBLE_CATEGORIES:
            status = "eligible"
            all_ineligible = False
            any_eligible = True
        else:
            status = "unknown"
            all_ineligible = False
            any_unknown = True
        lines.append(
            {
                "line_index": i,
                "category": li["category"],
                "status": status,
                "ineligible": status == "ineligible",
                "policy_ref": (
                    "POL-CAT-02"
                    if status == "ineligible"
                    else ("POL-CAT-01" if status == "eligible" else "POL-CAT-01")
                ),
            }
        )
    return {
        "claim_id": claim_id,
        "line_results": lines,
        "all_ineligible": all_ineligible and len(lines) > 0,
        "any_eligible": any_eligible,
        "any_unknown_category": any_unknown,
    }


def tool_check_receipts(claim_id: str) -> dict[str, Any]:
    claim = CLAIM_BY_ID.get(claim_id)
    if not claim:
        return {"error": f"Unknown claim_id {claim_id}"}
    missing: list[dict[str, Any]] = []
    for i, li in enumerate(claim["line_items"]):
        amt = money(li["amount"])
        cat = li["category"].lower()
        requires = cat in ALWAYS_RECEIPT_CATEGORIES or amt > LIMITS["receipt_threshold"]
        has = bool(li["receipt_attached"])
        entry = {
            "line_index": i,
            "category": li["category"],
            "amount": money_float(amt),
            "receipt_required": requires,
            "receipt_attached": has,
            "receipt_flag_sufficient_evidence": RECEIPT_ATTACHMENT_COUNTS_AS_EVIDENCE,
        }
        if requires and not has:
            missing.append(
                {
                    "line_index": i,
                    "category": li["category"],
                    "description": li["description"],
                    "doc_needed": f"{li['category']} itemized receipt",
                }
            )
        elif requires and has:
            entry["note"] = "Receipt flag Yes accepted as sufficient; no file/itemization supplied."
    return {
        "claim_id": claim_id,
        "missing_receipts": missing,
        "requires_manual_review": len(missing) > 0,
        "policy_refs": ["POL-RCT-01", "POL-RCT-02"],
    }


def allocate_pooled_daily_cap(
    pool: list[tuple[int, dict[str, Any], Decimal]],
    daily_cap: Decimal,
    policy_ref: str,
    category: str,
) -> tuple[list[dict[str, Any]], Decimal, Decimal]:
    """Split one combined daily cap across pooled same-day lines."""
    if not pool:
        return [], Decimal("0"), Decimal("0")
    total_claimed = sum(amt for _, _, amt in pool)
    total_approved = min(total_claimed, daily_cap)
    total_deducted = total_claimed - total_approved
    rows: list[dict[str, Any]] = []
    for i, li, amt in pool:
        if total_claimed > 0:
            share = amt / total_claimed
            line_approved = money(total_approved * share)
            line_deducted = amt - line_approved
        else:
            line_approved = Decimal("0")
            line_deducted = Decimal("0")
        rows.append(
            {
                "line_index": i,
                "category": li["category"],
                "claimed": money_float(amt),
                "approved_line": money_float(line_approved),
                "deducted_line": money_float(line_deducted),
                "daily_pool_aggregate": True,
                "policy_ref": policy_ref,
            }
        )
    return rows, total_approved, total_deducted


def explanation_contradicts_decision(explanation: str, decision: str) -> bool:
    text = explanation.lower()
    reject_markers = (
        "reject",
        "rejected",
        "no reimbursement",
        "not be paid",
        "not reimburs",
        "ineligible in full",
        "nothing reimbursable",
    )
    approve_markers = ("approve", "approved", "reimburs", "compliant", "eligible")
    if decision in ("APPROVE", "PARTIAL_APPROVE") and any(m in text for m in reject_markers):
        if not any(m in text for m in approve_markers):
            return True
        if "reject" in text or "no reimbursement" in text or "not be paid" in text:
            return True
    if decision == "REJECT" and any(
        m in text for m in ("fully approved", "approve the claim", "approved for", "no deductions needed")
    ):
        return True
    if decision == "MANUAL_REVIEW" and "fully approved for payment" in text:
        return True
    return False


def compose_evidence_aligned_explanation(
    decision: str,
    approved: float,
    deducted: float,
    pending: float,
    state: EvaluationState,
    claim: dict[str, Any],
) -> str:
    cid = claim["claim_id"]
    if decision == "APPROVE":
        return (
            f"{cid} is approved for ${approved:.2f} after policy checks "
            f"(deductions ${deducted:.2f}) based on tool evidence."
        )
    if decision == "PARTIAL_APPROVE":
        return (
            f"{cid} is partially approved: ${approved:.2f} reimbursable and ${deducted:.2f} "
            "deducted per per-diem and eligibility rules."
        )
    if decision == "REJECT":
        return f"{cid} is rejected; ${deducted:.2f} is deducted and ${approved:.2f} is reimbursable."
    return (
        f"{cid} requires manual review. ${pending:.2f} remains pending; "
        "approved_amount and deducted_amount are 0 until review completes."
    )


def request_explanation_correction(
    client: OpenAI,
    model: str,
    result: ClaimResult,
    state: EvaluationState,
    claim: dict[str, Any],
) -> tuple[str, float] | None:
    """One bounded correction request aligned to finalized decision and amounts."""
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "Rewrite the reimbursement explanation so it matches the finalized decision and "
                        "amounts exactly. Return only JSON: {\"explanation\": \"...\", \"confidence\": 0-1}."
                    ),
                },
                {
                    "role": "user",
                    "content": json.dumps(
                        {
                            "claim_id": result.claim_id,
                            "decision": result.decision,
                            "approved_amount": result.approved_amount,
                            "deducted_amount": result.deducted_amount,
                            "pending_amount": result.pending_amount,
                            "policy_refs": result.policy_refs,
                            "per_diem": state.per_diem,
                            "receipts": state.receipt_check,
                        }
                    ),
                },
            ],
        )
        content = (response.choices[0].message.content or "").strip()
        if content.startswith("```"):
            content = re.sub(r"^```(?:json)?\s*", "", content)
            content = re.sub(r"\s*```$", "", content)
        data = json.loads(content)
        expl = str(data.get("explanation", "")).strip()
        conf = float(data.get("confidence", result.confidence))
        if not expl:
            return None
        return expl, max(0.0, min(1.0, conf))
    except Exception:
        return None


def tool_calculate_per_diem(claim_id: str) -> dict[str, Any]:
    claim = CLAIM_BY_ID.get(claim_id)
    if not claim:
        return {"error": f"Unknown claim_id {claim_id}"}
    agg_notes: list[dict[str, Any]] = []
    line_results = []
    total_claimed = Decimal("0")
    total_approved = Decimal("0")
    total_deducted = Decimal("0")
    discrepancies: list[str] = list(claim.get("notes", []))

    trip_start = parse_date(claim["trip_start"])
    trip_end = parse_date(claim["trip_end"])
    calendar_nights = trip_calendar_nights(trip_start, trip_end)
    trip_days = max(1, (trip_end - trip_start).days + 1)
    single_calendar_day = trip_start == trip_end
    ground_pool: list[tuple[int, dict[str, Any], Decimal]] = []
    meal_pool: list[tuple[int, dict[str, Any], Decimal]] = []

    for i, li in enumerate(claim["line_items"]):
        cat = li["category"].lower()
        amt = money(li["amount"])
        total_claimed += amt
        if cat in INELIGIBLE_CATEGORIES:
            line_results.append(
                {
                    "line_index": i,
                    "category": li["category"],
                    "claimed": money_float(amt),
                    "approved": 0.0,
                    "deducted": money_float(amt),
                    "reason": "ineligible_category",
                    "policy_ref": "POL-CAT-02",
                }
            )
            total_deducted += amt
            continue
        if cat == "ground_transport":
            desc = li["description"]
            days_m = DAYS_RE.search(desc)
            explicit_days = int(days_m.group("n")) if days_m else 0
            has_rate_only = bool(RATE_RE.search(desc)) and not days_m
            if has_rate_only:
                meta, notes = parse_ground_transport_line(li, i, trip_start, trip_end)
                for n in notes:
                    agg_notes.append({"line_index": n.line_index, "kind": n.kind, "message": n.message})
                discrepancies.append(
                    f"Ground transport line {i}: daily allocation uncertain; manual review required."
                )
                line_results.append({**meta, "claimed": money_float(amt), "requires_manual_review": True})
                continue
            if trip_days > 1 and not days_m:
                discrepancies.append(
                    f"Ground transport line {i}: daily allocation uncertain; manual review required."
                )
                line_results.append(
                    {
                        "line_index": i,
                        "category": li["category"],
                        "claimed": money_float(amt),
                        "approved_line": 0.0,
                        "deducted_line": 0.0,
                        "requires_manual_review": True,
                        "policy_ref": "POL-PD-03",
                    }
                )
                continue
            if single_calendar_day and (not days_m or explicit_days <= 1):
                ground_pool.append((i, li, amt))
                continue
            meta, notes = parse_ground_transport_line(li, i, trip_start, trip_end)
            for n in notes:
                agg_notes.append({"line_index": n.line_index, "kind": n.kind, "message": n.message})
            if meta.get("requires_manual_review"):
                discrepancies.append(
                    f"Ground transport line {i}: ambiguous daily allocation requires manual review."
                )
            appr = money(meta["approved_line"])
            ded = money(meta["deducted_line"])
            line_results.append({**meta, "claimed": money_float(amt)})
            total_approved += appr
            total_deducted += ded
            continue
        if cat == "meals":
            desc = li["description"]
            days_m = DAYS_RE.search(desc)
            explicit_days = int(days_m.group("n")) if days_m else 0
            if single_calendar_day:
                if explicit_days > 1:
                    discrepancies.append(
                        f"Meals line {i}: multi-day description conflicts with single-day trip; manual review required."
                    )
                    line_results.append(
                        {
                            "line_index": i,
                            "category": li["category"],
                            "claimed": money_float(amt),
                            "approved_line": 0.0,
                            "deducted_line": 0.0,
                            "requires_manual_review": True,
                            "policy_ref": "POL-PD-01",
                        }
                    )
                    continue
                meal_pool.append((i, li, amt))
                continue
            meta, notes = parse_meals_lodging_line(li, i)
            for n in notes:
                agg_notes.append({"line_index": n.line_index, "kind": n.kind, "message": n.message})
            appr = money(meta["approved_line"])
            ded = money(meta["deducted_line"])
            line_results.append({**meta, "claimed": money_float(amt), "policy_ref": "POL-PD-01"})
            total_approved += appr
            total_deducted += ded
            continue
        if cat == "lodging":
            meta, notes = parse_meals_lodging_line(li, i)
            for n in notes:
                agg_notes.append({"line_index": n.line_index, "kind": n.kind, "message": n.message})
            if cat == "lodging" and "nights" in meta:
                stated = meta["nights"]
                if stated != calendar_nights and calendar_nights > 0:
                    discrepancies.append(
                        f"Lodging line states {stated} nights but trip dates imply {calendar_nights} calendar night(s)."
                    )
            appr = money(meta["approved_line"])
            ded = money(meta["deducted_line"])
            line_results.append({**meta, "claimed": money_float(amt), "policy_ref": f"POL-PD-0{'1' if cat=='meals' else '2'}"})
            total_approved += appr
            total_deducted += ded
            continue
        if cat == "airfare":
            cabin = classify_airfare_cabin(li["description"])
            if cabin == "economy":
                line_results.append(
                    {
                        "line_index": i,
                        "category": "airfare",
                        "claimed": money_float(amt),
                        "approved": money_float(amt),
                        "deducted": 0.0,
                        "cabin": cabin,
                        "policy_ref": "POL-AIR-01",
                    }
                )
                total_approved += amt
            else:
                line_results.append(
                    {
                        "line_index": i,
                        "category": "airfare",
                        "claimed": money_float(amt),
                        "approved_pending_review": money_float(amt),
                        "deducted": 0.0,
                        "cabin": cabin,
                        "airfare_exception": True,
                        "policy_ref": "POL-AIR-01",
                    }
                )
            continue
        if cat in ELIGIBLE_CATEGORIES:
            line_results.append(
                {
                    "line_index": i,
                    "category": li["category"],
                    "claimed": money_float(amt),
                    "approved": money_float(amt),
                    "deducted": 0.0,
                    "policy_ref": "POL-CAT-01",
                }
            )
            total_approved += amt
            continue
        line_results.append(
            {
                "line_index": i,
                "category": li["category"],
                "claimed": money_float(amt),
                "approved": 0.0,
                "deducted": 0.0,
                "unknown_category": True,
                "policy_ref": "POL-CAT-01",
            }
        )
        discrepancies.append(f"Unknown category '{li['category']}' on line {i}; requires manual review.")

    ground_rows, g_appr, g_ded = allocate_pooled_daily_cap(
        ground_pool, LIMITS["ground_per_day"], "POL-PD-03", "ground_transport"
    )
    if ground_rows:
        agg_notes.append(
            {
                "line_index": ground_rows[0]["line_index"],
                "kind": "ground_transport",
                "message": "Single-day trip: combined ground transport capped at $50/day (POL-PD-03).",
            }
        )
        line_results.extend(ground_rows)
        total_approved += g_appr
        total_deducted += g_ded

    meal_rows, m_appr, m_ded = allocate_pooled_daily_cap(
        meal_pool, LIMITS["meals_per_day"], "POL-PD-01", "meals"
    )
    if meal_rows:
        agg_notes.append(
            {
                "line_index": meal_rows[0]["line_index"],
                "kind": "meals",
                "message": "Single-day trip: combined meal spend capped at $75/day (POL-PD-01).",
            }
        )
        line_results.extend(meal_rows)
        total_approved += m_appr
        total_deducted += m_ded

    return {
        "claim_id": claim_id,
        "line_results": line_results,
        "total_claimed": money_float(total_claimed),
        "provisional_reimbursable_after_per_diem": money_float(total_approved),
        "provisional_deducted": money_float(total_deducted),
        "aggregation_notes": agg_notes,
        "trip_discrepancies": discrepancies,
        "calendar_nights_from_trip_dates": calendar_nights,
    }


def tool_check_timeliness(claim_id: str) -> dict[str, Any]:
    claim = CLAIM_BY_ID.get(claim_id)
    if not claim:
        return {"error": f"Unknown claim_id {claim_id}"}
    trip_start = parse_date(claim["trip_start"])
    trip_end = parse_date(claim["trip_end"])
    submitted = parse_date(claim["submitted_on"])
    within_start_window = submitted <= trip_start + timedelta(days=30)
    after_end_window = submitted > trip_end + timedelta(days=30)

    if within_start_window:
        status = "timely"
        detail = (
            "Submitted within 30 days of trip start; assuming expenses occurred within the trip interval, "
            "all expense dates in the window are timely. Advance purchases before trip start cannot be verified."
        )
        manual = False
    elif after_end_window:
        status = "late"
        detail = "Submitted more than 30 days after trip end; expenses treated as late for review."
        manual = True
    else:
        status = "uncertain"
        detail = (
            "Submission timing falls between trip-start+30 and trip-end+30 bounds; expense timing uncertain → review."
        )
        manual = True

    return {
        "claim_id": claim_id,
        "status": status,
        "submitted_on": claim["submitted_on"],
        "trip_start": claim["trip_start"],
        "trip_end": claim["trip_end"],
        "requires_manual_review": manual,
        "explanation": detail,
        "policy_ref": "POL-TIME-01",
    }


def tool_check_airfare_exception(claim_id: str) -> dict[str, Any]:
    claim = CLAIM_BY_ID.get(claim_id)
    if not claim:
        return {"error": f"Unknown claim_id {claim_id}"}
    exceptions = []
    for i, li in enumerate(claim["line_items"]):
        if li["category"].lower() != "airfare":
            continue
        cabin = classify_airfare_cabin(li["description"])
        if cabin == "business":
            exceptions.append(
                {
                    "line_index": i,
                    "description": li["description"],
                    "amount": money_float(money(li["amount"])),
                    "cabin": cabin,
                    "requires_manual_review": True,
                    "note": "Business/first class pending exception review; amount not auto-deducted.",
                }
            )
        elif cabin == "unknown":
            exceptions.append(
                {
                    "line_index": i,
                    "description": li["description"],
                    "amount": money_float(money(li["amount"])),
                    "cabin": cabin,
                    "requires_manual_review": True,
                    "note": "Airfare cabin class not specified; manual review required (POL-AIR-01).",
                }
            )
    return {
        "claim_id": claim_id,
        "exceptions": exceptions,
        "requires_manual_review": len(exceptions) > 0,
        "policy_ref": "POL-AIR-01",
    }


def tool_check_approval_tier(claim_id: str, state: EvaluationState) -> dict[str, Any]:
    if not state.per_diem:
        return {"error": "Run calculate_per_diem first", "claim_id": claim_id}
    reimbursable = Decimal(str(state.per_diem["provisional_reimbursable_after_per_diem"]))
    # Do not treat pending business airfare as certain for tier when exception unresolved
    if state.airfare_exception and state.airfare_exception.get("requires_manual_review"):
        tier_note = (
            "Approval threshold is conditional: business-class airfare amount remains pending exception review "
            "and is not treated as certain reimbursable for director-tier determination."
        )
        conditional = True
    else:
        tier_note = "Threshold evaluated on provisional reimbursable after per-diem per policy."
        conditional = False

    if reimbursable <= LIMITS["auto_approve_max"]:
        tier = "auto"
        pref = "POL-APR-01"
        director = False
    elif reimbursable <= LIMITS["manager_max"]:
        tier = "manager"
        pref = "POL-APR-02"
        director = False
    else:
        tier = "director"
        pref = "POL-APR-03"
        director = True

    pending_airfare_total = Decimal("0")
    if state.airfare_exception and state.airfare_exception.get("exceptions"):
        for ex in state.airfare_exception["exceptions"]:
            pending_airfare_total += money(Decimal(str(ex.get("amount", 0))))

    return {
        "claim_id": claim_id,
        "provisional_reimbursable": money_float(reimbursable),
        "pending_airfare_amount": money_float(pending_airfare_total),
        "provisional_if_airfare_approved": money_float(reimbursable + pending_airfare_total),
        "could_trigger_director_tier": money(reimbursable + pending_airfare_total) > LIMITS["manager_max"],
        "tier": tier,
        "policy_ref": pref,
        "requires_manual_review_director": director,
        "conditional_threshold": conditional,
        "explanation": tier_note,
        "tier_is_settled": not conditional,
    }


TOOL_IMPLS: dict[str, Callable[..., dict[str, Any]]] = {
    "lookup_policy": lambda rule_ids: tool_lookup_policy(rule_ids),
    "check_category_eligibility": lambda claim_id: tool_category_eligibility(claim_id),
    "check_receipts": lambda claim_id: tool_check_receipts(claim_id),
    "calculate_per_diem": lambda claim_id: tool_calculate_per_diem(claim_id),
    "check_timeliness": lambda claim_id: tool_check_timeliness(claim_id),
    "check_airfare_exception": lambda claim_id: tool_check_airfare_exception(claim_id),
}

OPENAI_TOOL_SPECS: list[dict[str, Any]] = [
    {
        "type": "function",
        "function": {
            "name": "lookup_policy",
            "description": "Return policy rule text for given POL-* identifiers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "rule_ids": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Policy rule ids e.g. POL-PD-01",
                    }
                },
                "required": ["rule_ids"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_category_eligibility",
            "description": "Check claim line categories against eligible/ineligible lists.",
            "parameters": {
                "type": "object",
                "properties": {"claim_id": {"type": "string"}},
                "required": ["claim_id"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_receipts",
            "description": "Determine required receipts and missing attachments for a claim.",
            "parameters": {
                "type": "object",
                "properties": {"claim_id": {"type": "string"}},
                "required": ["claim_id"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_per_diem",
            "description": "Compute per-diem deductions using canonical claim line items only.",
            "parameters": {
                "type": "object",
                "properties": {"claim_id": {"type": "string"}},
                "required": ["claim_id"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_timeliness",
            "description": "Evaluate claim submission timeliness using trip interval bounds.",
            "parameters": {
                "type": "object",
                "properties": {"claim_id": {"type": "string"}},
                "required": ["claim_id"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_airfare_exception",
            "description": "Detect business/first-class airfare requiring manual exception review.",
            "parameters": {
                "type": "object",
                "properties": {"claim_id": {"type": "string"}},
                "required": ["claim_id"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_approval_tier",
            "description": "Determine approval tier after per-diem; requires prior calculate_per_diem.",
            "parameters": {
                "type": "object",
                "properties": {"claim_id": {"type": "string"}},
                "required": ["claim_id"],
                "additionalProperties": False,
            },
        },
    },
]


def validate_tool_arguments(name: str, arguments: dict[str, Any]) -> dict[str, Any] | None:
    """Return error payload if invalid; None if OK."""
    if name == "lookup_policy":
        rule_ids = arguments.get("rule_ids")
        if not isinstance(rule_ids, list) or not rule_ids:
            return {"error": "rule_ids must be a non-empty array"}
        return None
    if name not in TOOL_IMPLS and name != "check_approval_tier":
        return {"error": f"Unknown tool {name}"}
    claim_id = arguments.get("claim_id")
    if not isinstance(claim_id, str) or claim_id not in CLAIM_BY_ID:
        return {"error": "claim_id must be one of the five provided claims"}
    return None


def json_tool_response(payload: dict[str, Any]) -> str:
    """Always return valid JSON; never slice raw JSON text."""
    text = json.dumps(payload, separators=(",", ":"))
    if len(text) <= MAX_TOOL_RESULT_CHARS:
        return text
    slim = dict(payload)
    if isinstance(slim.get("line_results"), list) and len(slim["line_results"]) > 5:
        slim = {
            **slim,
            "line_results": slim["line_results"][:5],
            "line_results_truncated": True,
            "line_results_total": len(payload["line_results"]),
        }
    text = json.dumps(slim, separators=(",", ":"))
    if len(text) > MAX_TOOL_RESULT_CHARS:
        keys = [k for k in slim.keys() if k != "line_results"][:12]
        summary = {k: slim[k] for k in keys}
        summary["truncated"] = True
        summary["note"] = "Payload shortened to fit context; essential scalar fields retained."
        text = json.dumps(summary, separators=(",", ":"))
    return text


def payload_success(payload: dict[str, Any] | None) -> bool:
    return isinstance(payload, dict) and "error" not in payload


def evidence_claim_id_ok(payload: dict[str, Any], state: EvaluationState) -> bool:
    cid = payload.get("claim_id")
    return cid is None or cid == state.claim_id


def tool_evidence_ok(tool_name: str, state: EvaluationState) -> bool:
    slot_map = {
        "check_category_eligibility": "category_eligibility",
        "check_receipts": "receipt_check",
        "calculate_per_diem": "per_diem",
        "check_timeliness": "timeliness",
        "check_airfare_exception": "airfare_exception",
        "check_approval_tier": "approval_tier",
    }
    attr = slot_map.get(tool_name)
    if not attr:
        return False
    data = getattr(state, attr)
    return payload_success(data) and evidence_claim_id_ok(data, state)


def missing_required_evidence(state: EvaluationState) -> set[str]:
    needed = required_tools_for_claim(state.claim_id)
    claim = CLAIM_BY_ID[state.claim_id]
    cat_state = state.category_eligibility
    if cat_state and cat_state.get("all_ineligible"):
        needed -= {"check_approval_tier", "check_airfare_exception"}
    if not any(li["category"].lower() == "airfare" for li in claim["line_items"]):
        needed.discard("check_airfare_exception")
    missing = {t for t in needed if not tool_evidence_ok(t, state)}
    return missing


def is_successful_llm_evaluation(result: ClaimResult) -> bool:
    """True when required claim-matched tool evidence is present (tools_used = successful tools only)."""
    if result.technical_failure:
        return False
    if not result.tools_used:
        return False
    state = EvaluationState(claim_id=result.claim_id)
    for tool in result.tools_used:
        if tool == "check_category_eligibility":
            state.category_eligibility = tool_category_eligibility(result.claim_id)
        elif tool == "check_receipts":
            state.receipt_check = tool_check_receipts(result.claim_id)
        elif tool == "calculate_per_diem":
            state.per_diem = tool_calculate_per_diem(result.claim_id)
        elif tool == "check_timeliness":
            state.timeliness = tool_check_timeliness(result.claim_id)
        elif tool == "check_airfare_exception":
            state.airfare_exception = tool_check_airfare_exception(result.claim_id)
        elif tool == "check_approval_tier":
            state.approval_tier = tool_check_approval_tier(result.claim_id, state)
    return len(missing_required_evidence(state)) == 0 and set(result.tools_used) >= (
        required_tools_for_claim(result.claim_id)
        - (
            {"check_approval_tier", "check_airfare_exception"}
            if state.category_eligibility and state.category_eligibility.get("all_ineligible")
            else set()
        )
    )


def execute_tool(
    name: str,
    arguments: dict[str, Any],
    state: EvaluationState,
    audit: list[dict[str, Any]],
    source: Literal["llm", "system"] = "llm",
) -> tuple[str, bool]:
    payload: dict[str, Any]

    rejected_reason: str | None = None
    arg_err = validate_tool_arguments(name, arguments)
    if arg_err:
        payload = arg_err
        rejected_reason = "invalid_arguments"
    elif name in CLAIM_SCOPED_TOOLS and arguments.get("claim_id") != state.claim_id:
        payload = {
            "error": "claim_id does not match active evaluation state",
            "expected": state.claim_id,
            "received": arguments.get("claim_id"),
        }
        rejected_reason = "claim_id_mismatch"
    elif name not in TOOL_IMPLS and name != "check_approval_tier":
        payload = {"error": f"Unknown tool {name}"}
        rejected_reason = "unknown_tool"
    elif name == "lookup_policy":
        payload = tool_lookup_policy(arguments.get("rule_ids", []))
        if payload_success(payload):
            state.policy_lookups.extend(arguments.get("rule_ids", []))
    elif name == "check_category_eligibility":
        payload = tool_category_eligibility(arguments["claim_id"])
    elif name == "check_receipts":
        payload = tool_check_receipts(arguments["claim_id"])
    elif name == "calculate_per_diem":
        payload = tool_calculate_per_diem(arguments["claim_id"])
    elif name == "check_timeliness":
        payload = tool_check_timeliness(arguments["claim_id"])
    elif name == "check_airfare_exception":
        payload = tool_check_airfare_exception(arguments["claim_id"])
    elif name == "check_approval_tier":
        payload = tool_check_approval_tier(state.claim_id, state)
    else:
        payload = {"error": f"Unhandled tool {name}"}
        rejected_reason = "unhandled_tool"

    success = payload_success(payload) and evidence_claim_id_ok(payload, state)
    if success:
        if name == "check_category_eligibility":
            state.category_eligibility = payload
        elif name == "check_receipts":
            state.receipt_check = payload
        elif name == "calculate_per_diem":
            state.per_diem = payload
        elif name == "check_timeliness":
            state.timeliness = payload
        elif name == "check_airfare_exception":
            state.airfare_exception = payload
        elif name == "check_approval_tier":
            state.approval_tier = payload

    text = json_tool_response(payload if isinstance(payload, dict) else {"result": payload})
    preview = json.loads(text)
    if len(text) > 2000:
        preview_text = json.dumps(
            {"preview_note": "audit preview shortened", "keys": list(preview.keys())[:15]},
            separators=(",", ":"),
        )
    else:
        preview_text = text
    audit.append(
        {
            "tool": name,
            "source": source,
            "arguments": arguments,
            "success": success,
            "rejected": rejected_reason,
            "result_preview": preview_text,
            "result_text": text,
        }
    )
    return text, success


def required_tools_for_claim(claim_id: str) -> set[str]:
    base = {
        "check_category_eligibility",
        "check_receipts",
        "calculate_per_diem",
        "check_timeliness",
    }
    claim = CLAIM_BY_ID[claim_id]
    if any(li["category"].lower() == "airfare" for li in claim["line_items"]):
        base.add("check_airfare_exception")
    base.add("check_approval_tier")
    return base


def missing_required_tools(state: EvaluationState, _attempted: set[str] | None = None) -> set[str]:
    """Backward-compatible alias; ignores attempted names and inspects evidence state only."""
    return missing_required_evidence(state)


def canonical_missing_docs(state: EvaluationState) -> list[str]:
    docs: list[str] = []
    seen: set[tuple[int, str]] = set()
    if not state.receipt_check:
        return docs
    for m in state.receipt_check.get("missing_receipts", []):
        key = (int(m["line_index"]), str(m["category"]).lower())
        if key in seen:
            continue
        seen.add(key)
        doc = m.get("doc_needed")
        if doc:
            docs.append(doc)
    return docs


# ---------------------------------------------------------------------------
# Deterministic finalize
# ---------------------------------------------------------------------------


def deterministic_manual_triggers(state: EvaluationState, claim: dict[str, Any]) -> list[str]:
    reasons: list[str] = []
    if state.receipt_check and state.receipt_check.get("requires_manual_review"):
        reasons.append("missing_required_receipt")
    if state.category_eligibility and state.category_eligibility.get("any_unknown_category"):
        reasons.append("unknown_category")
    if state.airfare_exception and state.airfare_exception.get("requires_manual_review"):
        if any(ex.get("cabin") == "unknown" for ex in state.airfare_exception.get("exceptions", [])):
            reasons.append("unknown_airfare_cabin")
        if any(ex.get("cabin") == "business" for ex in state.airfare_exception.get("exceptions", [])):
            reasons.append("business_class_airfare")
    if state.timeliness and state.timeliness.get("requires_manual_review"):
        reasons.append("timeliness_uncertain_or_late")
    if state.per_diem:
        for d in state.per_diem.get("trip_discrepancies", []):
            reasons.append(f"data_discrepancy: {d}")
    if state.approval_tier and state.approval_tier.get("requires_manual_review_director"):
        if not state.approval_tier.get("conditional_threshold"):
            reasons.append("director_threshold")
        else:
            reasons.append("director_threshold_conditional")
    return reasons


def compute_final_amounts(
    decision: str,
    state: EvaluationState,
    claim: dict[str, Any],
    manual_forced: bool,
) -> tuple[float, float, float, list[str]]:
    """Returns approved, deducted, pending, safety_messages."""
    total = claim_total(claim)
    safety: list[str] = []
    if decision == "MANUAL_REVIEW" or manual_forced:
        pending = money_float(total)
        return 0.0, 0.0, pending, safety

    if not state.per_diem or not state.category_eligibility:
        safety.append("Missing tool evidence for amounts; forced MANUAL_REVIEW.")
        return 0.0, 0.0, money_float(total), safety

    if state.category_eligibility.get("all_ineligible"):
        return 0.0, money_float(total), 0.0, safety

    appr = Decimal(str(state.per_diem["provisional_reimbursable_after_per_diem"]))
    ded = Decimal(str(state.per_diem["provisional_deducted"]))
    total_dec = money(appr + ded)
    if total_dec != money(total):
        # conference fees etc. should reconcile
        if money(appr + ded) != money(total):
            safety.append(f"Amount reconciliation mismatch: approved+deducted={appr+ded} vs claimed={total}")

    if decision == "REJECT":
        return 0.0, money_float(total), 0.0, safety
    if decision == "APPROVE":
        if ded > 0:
            safety.append("Approve requested with deductions; downgraded to PARTIAL_APPROVE.")
        return money_float(appr), money_float(ded), 0.0, safety
    if decision == "PARTIAL_APPROVE":
        return money_float(appr), money_float(ded), 0.0, safety
    return 0.0, 0.0, money_float(total), safety


def append_policy_citation_notes(
    decision: str,
    policy_refs: list[str],
    explanation: str,
    state: EvaluationState,
    manual_triggers: list[str],
) -> tuple[list[str], str]:
    """Deterministic citation/explanation consistency from tool evidence (not claim-id keyed)."""
    refs = list(dict.fromkeys(policy_refs))
    note_parts: list[str] = []

    if state.approval_tier and state.approval_tier.get("conditional_threshold"):
        for apr in ("POL-APR-01", "POL-APR-02", "POL-APR-03"):
            while apr in refs:
                refs.remove(apr)
        prov = state.approval_tier.get("provisional_reimbursable", 0)
        note_parts.append(
            f"Pending (POL-AIR-01): business-class airfare eligibility unresolved; "
            f"no settled approval tier. Known non-airfare reimbursable after per-diem: ${prov:.2f}."
        )
        if state.approval_tier.get("could_trigger_director_tier"):
            if "POL-APR-03" not in refs:
                refs.append("POL-APR-03")
            note_parts.append(
                "Conditional (POL-APR-03): if the airfare exception were approved, post-deduction total "
                f"could reach ${state.approval_tier.get('provisional_if_airfare_approved', 0):.2f} and exceed $2,000."
            )

    if state.per_diem:
        for line in state.per_diem.get("line_results", []):
            cat = str(line.get("category", "")).lower()
            ded = float(line.get("deducted_line") or line.get("deducted") or 0)
            if decision == "PARTIAL_APPROVE" and cat == "lodging" and ded > 0:
                if "POL-PD-02" not in refs:
                    refs.append("POL-PD-02")
                note_parts.append(
                    f"Lodging adjusted under POL-PD-02 ($200/night maximum); ${ded:.2f} deducted from claimed lodging."
                )
            if cat == "meals" and (
                ded > 0 or line.get("single_line_meal") or manual_triggers
            ):
                prov = float(line.get("approved_line", 0))
                if "POL-PD-01" not in refs:
                    refs.append("POL-PD-01")
                if decision == "MANUAL_REVIEW" and state.receipt_check and state.receipt_check.get(
                    "requires_manual_review"
                ):
                    note_parts.append(
                        f"Provisional only (POL-PD-01, $75/day): up to ${prov:.2f} reimbursable and "
                        f"${ded:.2f} excess if receipt were satisfied—not authorized pending review."
                    )
                elif ded > 0:
                    note_parts.append(
                        f"Meals capped under POL-PD-01 ($75/day); ${ded:.2f} deducted above daily limit."
                    )

    if note_parts:
        explanation = explanation + " Evidence notes: " + " ".join(note_parts)
    return refs, explanation


def validate_finalize(
    rec: LLMRecommendation,
    state: EvaluationState,
    claim: dict[str, Any],
    tools_succeeded: set[str],
    audit: list[dict[str, Any]],
) -> ClaimResult:
    safety_overrides: list[str] = []
    decision = rec.decision
    manual_triggers = deterministic_manual_triggers(state, claim)
    missing_tools = missing_required_evidence(state)

    minimal_reject_evidence = {
        "check_category_eligibility",
        "calculate_per_diem",
        "check_receipts",
        "check_timeliness",
    }
    if (
        state.category_eligibility
        and state.category_eligibility.get("all_ineligible")
        and not manual_triggers
        and all(tool_evidence_ok(t, state) for t in minimal_reject_evidence)
    ):
        if decision != "REJECT":
            safety_overrides.append(
                "All lines ineligible with core evidence gathered; forcing REJECT (approval tier not required)."
            )
        decision = "REJECT"
        missing_tools = set()

    if missing_tools:
        safety_overrides.append(
            f"Insufficient tool evidence (missing: {sorted(missing_tools)}); cannot finalize approval."
        )
        decision = "MANUAL_REVIEW"

    if manual_triggers:
        if decision not in ("MANUAL_REVIEW",):
            safety_overrides.append(
                f"Manual review triggers present ({manual_triggers}); overriding {rec.decision}."
            )
        decision = "MANUAL_REVIEW"

    # Ineligible-only claims resolve to REJECT (not MR) when no policy-exception triggers fired.
    if (
        state.category_eligibility
        and state.category_eligibility.get("all_ineligible")
        and not manual_triggers
        and not missing_tools
    ):
        if decision != "REJECT":
            safety_overrides.append("All lines ineligible; forcing REJECT.")
        decision = "REJECT"
    elif state.category_eligibility and state.category_eligibility.get("all_ineligible"):
        if decision not in ("REJECT", "MANUAL_REVIEW"):
            safety_overrides.append("All lines ineligible; forcing REJECT.")
        if decision != "MANUAL_REVIEW":
            decision = "REJECT"

    if decision == "REJECT" and state.category_eligibility:
        all_ineligible = state.category_eligibility.get("all_ineligible")
        reimbursable = (
            float(state.per_diem.get("provisional_reimbursable_after_per_diem", 0) or 0)
            if state.per_diem
            else 0.0
        )
        if not all_ineligible and reimbursable > 0:
            safety_overrides.append(
                "REJECT unsupported: eligible reimbursable value remains after policy tools; correcting outcome."
            )
            if state.per_diem and float(state.per_diem.get("provisional_deducted", 0) or 0) > 0:
                decision = "PARTIAL_APPROVE"
            else:
                decision = "APPROVE"

    manual_forced = decision == "MANUAL_REVIEW" and (
        bool(manual_triggers) or bool(missing_tools) or rec.decision == "MANUAL_REVIEW"
    )
    approved, deducted, pending, amt_safety = compute_final_amounts(decision, state, claim, manual_forced)
    safety_overrides.extend(amt_safety)

    if amt_safety and decision != "MANUAL_REVIEW":
        decision = "MANUAL_REVIEW"
        approved, deducted, pending = 0.0, 0.0, money_float(claim_total(claim))
        safety_overrides.append("Amount validation failed; conservative MANUAL_REVIEW.")

    if decision in ("APPROVE", "PARTIAL_APPROVE", "REJECT"):
        total = money_float(claim_total(claim))
        if abs((approved + deducted) - total) > 0.02 and decision == "REJECT" and approved == 0:
            pass
        elif decision in ("APPROVE", "PARTIAL_APPROVE"):
            if abs((approved + deducted) - total) > 0.02:
                safety_overrides.append("Financial reconciliation failed; MANUAL_REVIEW.")
                decision = "MANUAL_REVIEW"
                approved, deducted, pending = 0.0, 0.0, total

    if decision == "PARTIAL_APPROVE" and deducted <= 0:
        safety_overrides.append("PARTIAL_APPROVE without deductions; changing to APPROVE.")
        decision = "APPROVE"

    if decision == "APPROVE" and deducted > 0:
        decision = "PARTIAL_APPROVE"

    policy_refs = list(dict.fromkeys(rec.policy_refs))
    for trigger in manual_triggers:
        if "receipt" in trigger and "POL-RCT-02" not in policy_refs:
            policy_refs.append("POL-RCT-02")
        if "business_class" in trigger and "POL-AIR-01" not in policy_refs:
            policy_refs.append("POL-AIR-01")
        if "timeliness" in trigger and "POL-TIME-01" not in policy_refs:
            policy_refs.append("POL-TIME-01")
        if "director" in trigger and "POL-APR-03" not in policy_refs:
            policy_refs.append("POL-APR-03")
    if state.category_eligibility and state.category_eligibility.get("all_ineligible"):
        if "POL-CAT-02" not in policy_refs:
            policy_refs.append("POL-CAT-02")
    if decision == "APPROVE" and "POL-APR-01" not in policy_refs and "POL-APR-02" not in policy_refs:
        if state.approval_tier:
            policy_refs.append(state.approval_tier.get("policy_ref", "POL-APR-02"))

    missing_docs = canonical_missing_docs(state)

    audit.append({"event": "original_recommendation", "recommendation": rec.model_dump()})

    explanation = rec.explanation.strip()
    decision_changed = decision != rec.decision
    if decision_changed or explanation_contradicts_decision(explanation, decision):
        explanation = compose_evidence_aligned_explanation(
            decision, approved, deducted, pending, state, claim
        )
        if decision == "MANUAL_REVIEW":
            if state.per_diem:
                prov = state.per_diem.get("provisional_reimbursable_after_per_diem")
                prov_d = state.per_diem.get("provisional_deducted")
                explanation += (
                    f" Provisional calc only (not final): reimbursable≈${prov:.2f}, deductions≈${prov_d:.2f}."
                )

    confidence = float(rec.confidence)

    policy_refs, explanation = append_policy_citation_notes(
        decision, policy_refs, explanation, state, manual_triggers
    )

    audit.append({"event": "validate_finalize", "source": "system", "safety_overrides": safety_overrides})

    return ClaimResult(
        claim_id=claim["claim_id"],
        decision=decision,  # type: ignore[arg-type]
        approved_amount=approved,
        deducted_amount=deducted,
        missing_docs=missing_docs,
        policy_refs=policy_refs,
        confidence=confidence,
        explanation=explanation,
        tools_used=sorted(tools_succeeded),
        pending_amount=pending,
        audit_trail=audit,
        safety_overrides=safety_overrides,
    )


# ---------------------------------------------------------------------------
# LLM agent loop
# ---------------------------------------------------------------------------


SYSTEM_PROMPT = textwrap.dedent(
    """
    You are a travel reimbursement approval agent. Use tools to gather evidence from canonical claim data.
    Never invent amounts, receipts, or approvals. Receipt flags of Yes are sufficient mock evidence (no OCR).
    After sufficient tool calls, respond with ONLY a JSON object matching:
    {
      "claim_id": "...",
      "decision": "APPROVE|PARTIAL_APPROVE|REJECT|MANUAL_REVIEW",
      "confidence": 0.0-1.0,
      "explanation": "...",
      "policy_refs": ["POL-..."],
      "missing_docs": []
    }
    Prefer MANUAL_REVIEW for missing receipts, business-class airfare, director tier when certain,
    timeliness uncertainty, and data discrepancies. For MANUAL_REVIEW do not assert final payment amounts.
    Confidence is your confidence in the routing recommendation (not payment probability).
    """
).strip()


class ConfigurationError(RuntimeError):
    pass


def get_openai_client() -> OpenAI:
    key = os.environ.get("OPENAI_API_KEY")
    if not key:
        raise ConfigurationError(
            "OPENAI_API_KEY is not set. Export the key or add it to a local .env file (not committed)."
        )
    return OpenAI(api_key=key, timeout=OPENAI_TIMEOUT_SECONDS)


def call_hash(name: str, args: dict[str, Any]) -> str:
    blob = json.dumps({"name": name, "args": args}, sort_keys=True)
    return hashlib.sha256(blob.encode()).hexdigest()[:16]


def run_agent_for_claim(client: OpenAI, claim_id: str, model: str) -> ClaimResult:
    claim = CLAIM_BY_ID[claim_id]
    state = EvaluationState(claim_id=claim_id)
    audit: list[dict[str, Any]] = []
    tools_succeeded: set[str] = set()
    seen_success_hashes: set[str] = set()

    messages: list[dict[str, Any]] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Evaluate claim {claim_id} for employee {claim['employee']}. "
                f"Call the necessary tools (use claim_id only). Then return final JSON recommendation."
            ),
        },
    ]

    recommendation: LLMRecommendation | None = None
    last_validation_error: str | None = None

    for iteration in range(MAX_AGENT_ITERATIONS):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                tools=OPENAI_TOOL_SPECS,
                tool_choice="auto",
            )
        except Exception as exc:  # noqa: BLE001 — provider failure fallback
            audit.append({"event": "provider_failure", "error": str(exc)})
            rec = LLMRecommendation(
                claim_id=claim_id,
                decision="MANUAL_REVIEW",
                confidence=0.0,
                explanation=(
                    "Evaluation incomplete due to technical failure contacting the model provider. "
                    "No policy decision was finalized."
                ),
                policy_refs=[],
                missing_docs=[],
            )
            result = validate_finalize(rec, state, claim, tools_succeeded, audit)
            result.technical_failure = True
            result.explanation += f" Provider error: {type(exc).__name__}."
            return result
        choice = response.choices[0]
        msg = choice.message

        if choice.finish_reason == "tool_calls" and msg.tool_calls:
            messages.append(
                {
                    "role": "assistant",
                    "content": msg.content,
                    "tool_calls": [
                        {
                            "id": tc.id,
                            "type": tc.type,
                            "function": {
                                "name": tc.function.name,
                                "arguments": tc.function.arguments,
                            },
                        }
                        for tc in msg.tool_calls
                    ],
                }
            )
            for tc in msg.tool_calls:
                fname = tc.function.name
                try:
                    fargs = json.loads(tc.function.arguments or "{}")
                except json.JSONDecodeError:
                    result = json.dumps({"error": "Invalid JSON arguments"})
                    audit.append({"tool": fname, "source": "llm", "error": "invalid_args"})
                else:
                    h = call_hash(fname, fargs)
                    if h in seen_success_hashes:
                        dup_payload = {"error": "Duplicate tool call suppressed", "prior_hash": h, "cached": True}
                        result = json.dumps(dup_payload)
                        audit.append(
                            {
                                "tool": fname,
                                "source": "llm",
                                "arguments": fargs,
                                "duplicate_suppressed": True,
                                "success": False,
                                "result_text": result,
                            }
                        )
                    else:
                        result, ok = execute_tool(fname, fargs, state, audit, source="llm")
                        if ok:
                            seen_success_hashes.add(h)
                            tools_succeeded.add(fname)
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
            continue

        content = (msg.content or "").strip()
        if not content:
            messages.append({"role": "user", "content": "Provide the final JSON recommendation."})
            continue

        # Strip markdown fences if present
        if content.startswith("```"):
            content = re.sub(r"^```(?:json)?\s*", "", content)
            content = re.sub(r"\s*```$", "", content)

        for attempt in range(MAX_JSON_DECODE_RETRIES + 1):
            try:
                data = json.loads(content)
                recommendation = LLMRecommendation.model_validate(data)
                if recommendation.claim_id != claim_id:
                    raise ValueError("claim_id mismatch in recommendation")
                break
            except (json.JSONDecodeError, ValidationError, ValueError) as exc:
                last_validation_error = str(exc)
                if attempt < MAX_JSON_DECODE_RETRIES:
                    messages.append({"role": "assistant", "content": content})
                    messages.append(
                        {
                            "role": "user",
                            "content": f"JSON invalid: {exc}. Return corrected JSON only.",
                        }
                    )
                    try:
                        response2 = client.chat.completions.create(
                            model=model, messages=messages, tools=OPENAI_TOOL_SPECS
                        )
                    except Exception as exc:  # noqa: BLE001
                        audit.append({"event": "provider_failure", "error": str(exc)})
                        last_validation_error = str(exc)
                        break
                    content = (response2.choices[0].message.content or "").strip()
                    if content.startswith("```"):
                        content = re.sub(r"^```(?:json)?\s*", "", content)
                        content = re.sub(r"\s*```$", "", content)
                    continue
                break
        if recommendation:
            still_missing = missing_required_evidence(state)
            if still_missing:
                messages.append({"role": "assistant", "content": content})
                messages.append(
                    {
                        "role": "user",
                        "content": (
                            "Required evidence tools not yet called. "
                            f"Call these tools with claim_id only, then return JSON again: {sorted(still_missing)}"
                        ),
                    }
                )
                recommendation = None
                continue
            break
        messages.append({"role": "assistant", "content": content})
        messages.append({"role": "user", "content": "Return valid JSON recommendation after using required tools."})

    if recommendation is None:
        audit.append(
            {
                "event": "agent_failure",
                "error": last_validation_error or "no recommendation",
                "reason": "iteration_limit_or_invalid_output",
            }
        )
        rec = LLMRecommendation(
            claim_id=claim_id,
            decision="MANUAL_REVIEW",
            confidence=0.0,
            explanation="Evaluation incomplete: agent did not return a valid structured recommendation.",
            policy_refs=[],
            missing_docs=[],
        )
        result = validate_finalize(rec, state, claim, tools_succeeded, audit)
        result.technical_failure = True
        result.explanation += " Technical failure: invalid or missing LLM JSON or iteration limit."
        return result

    result = validate_finalize(recommendation, state, claim, tools_succeeded, audit)
    if explanation_contradicts_decision(result.explanation, result.decision):
        corrected = request_explanation_correction(client, model, result, state, claim)
        if corrected:
            new_expl, new_conf = corrected
            if not explanation_contradicts_decision(new_expl, result.decision):
                result.explanation = new_expl
                result.confidence = new_conf
            else:
                result.decision = "MANUAL_REVIEW"  # type: ignore[assignment]
                result.approved_amount = 0.0
                result.deducted_amount = 0.0
                result.pending_amount = money_float(claim_total(claim))
                result.explanation = (
                    "Unresolved conflict between model recommendation and policy evidence after correction; "
                    "route to manual review."
                )
                result.confidence = 0.0
                result.safety_overrides.append("explanation_decision_conflict_unresolved")
        elif explanation_contradicts_decision(result.explanation, result.decision):
            result.decision = "MANUAL_REVIEW"  # type: ignore[assignment]
            result.approved_amount = 0.0
            result.deducted_amount = 0.0
            result.pending_amount = money_float(claim_total(claim))
            result.explanation = (
                "Unresolved conflict between model recommendation and policy evidence; "
                "explanation correction unavailable—manual review required."
            )
            result.confidence = 0.0
            result.safety_overrides.append("explanation_decision_conflict_unresolved")
    if missing_required_evidence(state):
        if not result.safety_overrides:
            result.safety_overrides.append("Required evidence tools incomplete.")
        if "insufficient tool evidence" not in result.explanation.lower():
            result.explanation += " Required policy/evidence tools were not all invoked successfully."
    return result


def evaluate_all_claims(client: OpenAI | None = None, model: str | None = None) -> list[ClaimResult]:
    model = model or DEFAULT_OPENAI_MODEL
    if client is None:
        client = get_openai_client()
    results: list[ClaimResult] = []
    for claim in CLAIMS:
        results.append(run_agent_for_claim(client, claim["claim_id"], model))
    return results


def results_to_export_json(results: list[ClaimResult]) -> list[dict[str, Any]]:
    out = []
    for r in results:
        out.append(
            {
                "claim_id": r.claim_id,
                "decision": r.decision,
                "approved_amount": r.approved_amount,
                "deducted_amount": r.deducted_amount,
                "missing_docs": r.missing_docs,
                "policy_refs": r.policy_refs,
                "confidence": round(r.confidence, 3),
                "explanation": r.explanation,
                "tools_used": r.tools_used,
            }
        )
    return out

## Sample claims (Appendix B)

The five claims are defined in the `CLAIMS` list in the code cell above (faithful transcription of the PDF).

In [3]:
import json
from IPython.display import display, Markdown, HTML

# Shared evaluation state for dashboard + export (single source of truth)
EVAL_RESULTS: list[ClaimResult] = []
EVAL_ERROR: str | None = None
EVAL_BUSY: bool = False
EVAL_MODEL_USED: str | None = None
EVAL_LAST_RUN_SUCCESS: bool = False
EVAL_RUN_ID: str | None = None
EVAL_RUN_AT: str | None = None


def validate_result_set(results: list[ClaimResult]) -> None:
    if len(results) != 5:
        raise ValueError(f"Expected 5 results, got {len(results)}")
    ids = {r.claim_id for r in results}
    expected = {c["claim_id"] for c in CLAIMS}
    if ids != expected:
        raise ValueError(f"Claim id mismatch: {ids} vs {expected}")
    for r in results:
        if r.decision not in DECISION_VALUES:
            raise ValueError(r.decision)
        for p in r.policy_refs:
            if p not in VALID_POLICY_IDS:
                raise ValueError(p)
        if any(x < 0 for x in [r.approved_amount, r.deducted_amount, r.pending_amount]):
            raise ValueError("negative money")
        if r.decision in ("APPROVE", "PARTIAL_APPROVE", "REJECT"):
            total = money_float(claim_total(CLAIM_BY_ID[r.claim_id]))
            if abs((r.approved_amount + r.deducted_amount) - total) > 0.02:
                raise ValueError(f"Reconciliation failed for {r.claim_id}")
        if r.decision == "MANUAL_REVIEW":
            if r.approved_amount != 0 or r.deducted_amount != 0:
                raise ValueError("MANUAL_REVIEW must be 0/0 amounts")
        if r.decision == "PARTIAL_APPROVE" and r.deducted_amount <= 0:
            raise ValueError("PARTIAL_APPROVE needs deductions")
        if r.decision == "APPROVE" and r.deducted_amount > 0:
            raise ValueError("APPROVE cannot have deductions")


def assert_live_evaluation_success(results: list[ClaimResult]) -> None:
    incomplete = [r.claim_id for r in results if not is_successful_llm_evaluation(r)]
    if incomplete:
        technical = [r.claim_id for r in results if r.technical_failure]
        raise RuntimeError(
            "Incomplete LLM policy evaluation for "
            f"{', '.join(incomplete)}. Technical failures: {', '.join(technical) or 'none'}."
        )


def run_evaluation_once() -> list[ClaimResult]:
    global EVAL_RESULTS, EVAL_ERROR, EVAL_BUSY, EVAL_MODEL_USED, EVAL_LAST_RUN_SUCCESS
    global EVAL_RUN_ID, EVAL_RUN_AT
    if EVAL_BUSY:
        raise RuntimeError("Evaluation already in progress")
    EVAL_BUSY = True
    EVAL_ERROR = None
    EVAL_LAST_RUN_SUCCESS = False
    EVAL_RUN_ID = None
    EVAL_RUN_AT = None
    try:
        client = get_openai_client()
        model = DEFAULT_OPENAI_MODEL
        EVAL_MODEL_USED = model
        EVAL_RESULTS = evaluate_all_claims(client=client, model=model)
        validate_result_set(EVAL_RESULTS)
        assert_live_evaluation_success(EVAL_RESULTS)
        from datetime import datetime, timezone
        import uuid

        EVAL_RUN_ID = uuid.uuid4().hex[:10]
        EVAL_RUN_AT = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        EVAL_LAST_RUN_SUCCESS = True
        return EVAL_RESULTS
    except ConfigurationError as exc:
        EVAL_ERROR = str(exc)
        EVAL_RESULTS = []
        raise
    except Exception as exc:  # noqa: BLE001 — surfaced to notebook UI
        EVAL_ERROR = f"{type(exc).__name__}: {exc}"
        EVAL_RESULTS = []
        raise
    finally:
        EVAL_BUSY = False


import html


def escape_html(text: str) -> str:
    return html.escape(str(text), quote=True)


def evaluation_snapshot_banner() -> str:
    if EVAL_LAST_RUN_SUCCESS and EVAL_RUN_ID and EVAL_RUN_AT:
        return (
            f"<p><b>Current evaluation snapshot</b> "
            f"<code>{escape_html(EVAL_RUN_ID)}</code> at {escape_html(EVAL_RUN_AT)}</p>"
        )
    if EVAL_ERROR:
        return f"<p style='color:#b00020'><b>Incomplete run</b> — {escape_html(EVAL_ERROR)}</p>"
    return "<p><i>No successful evaluation snapshot</i></p>"


def render_static_summary_html(results: list[ClaimResult]) -> str:
    banner = evaluation_snapshot_banner()
    if not results:
        return banner + "<p><i>No results to display.</i></p>"
    counts: dict[str, int] = {}
    appr = ded = pend = 0.0
    for r in results:
        counts[r.decision] = counts.get(r.decision, 0) + 1
        appr += r.approved_amount
        ded += r.deducted_amount
        pend += r.pending_amount
    rows = "".join(
        f"<tr><td>{escape_html(r.claim_id)}</td><td>{escape_html(r.decision)}</td>"
        f"<td style='text-align:right'>{r.approved_amount:.2f}</td>"
        f"<td style='text-align:right'>{r.deducted_amount:.2f}</td>"
        f"<td style='text-align:right'>{r.pending_amount:.2f}</td></tr>"
        for r in sorted(results, key=lambda x: x.claim_id)
    )
    return (
        f"{banner}"
        "<h3>Results summary</h3>"
        "<table border='1' cellpadding='6' cellspacing='0' style='border-collapse:collapse'>"
        "<tr><th>Metric</th><th style='text-align:right'>Value</th></tr>"
        f"<tr><td>Decisions</td><td style='text-align:right'>{len(results)}</td></tr>"
        f"<tr><td>Total approved (finalized)</td><td style='text-align:right'>{appr:,.2f}</td></tr>"
        f"<tr><td>Total deducted (finalized)</td><td style='text-align:right'>{ded:,.2f}</td></tr>"
        f"<tr><td>Total pending (manual review)</td><td style='text-align:right'>{pend:,.2f}</td></tr>"
        "</table>"
        f"<p><b>Decision counts:</b> {escape_html(', '.join(f'{k}={v}' for k, v in sorted(counts.items())))}</p>"
        "<table border='1' cellpadding='6' cellspacing='0' style='border-collapse:collapse'>"
        "<tr><th>Claim</th><th>Decision</th><th>Approved</th><th>Deducted</th><th>Pending</th></tr>"
        f"{rows}</table>"
    )


from IPython.display import JSON


def render_sample_outputs(results: list[ClaimResult]) -> None:
    if not EVAL_LAST_RUN_SUCCESS or len(results) < 3:
        display(Markdown("*Sample outputs require a successful evaluation snapshot (Run All or dashboard run).*"))
        return
    stamp = f"`{EVAL_RUN_ID}` @ {EVAL_RUN_AT}"
    display(Markdown(f"## Sample outputs (snapshot {stamp})"))
    display(Markdown(f"_Historical notebook outputs may show older snapshots; only a successful current run is authoritative._"))
    for r in results[:3]:
        display(Markdown(f"### {r.claim_id} — {r.decision}"))
        display(
            JSON(
                {
                    "evaluation_run_id": EVAL_RUN_ID,
                    "evaluation_run_at": EVAL_RUN_AT,
                    "claim_id": r.claim_id,
                    "decision": r.decision,
                    "approved_amount": r.approved_amount,
                    "deducted_amount": r.deducted_amount,
                    "pending_amount": r.pending_amount,
                    "policy_refs": r.policy_refs,
                    "confidence": r.confidence,
                    "explanation": r.explanation[:1200],
                    "tools_used": r.tools_used,
                }
            )
        )


def render_audit_html(audit_trail: list) -> str:
    parts: list[str] = []
    for i, entry in enumerate(audit_trail):
        if entry.get("event") == "validate_finalize":
            parts.append(
                "<details open><summary><b>system</b> validate_finalize</summary>"
                f"<pre>{escape_html(json.dumps(entry.get('safety_overrides', []), indent=2))}</pre></details>"
            )
            continue
        if entry.get("event") == "provider_failure":
            parts.append(
                f"<details open><summary><b>provider_failure</b></summary>"
                f"<pre>{escape_html(json.dumps(entry, indent=2))}</pre></details>"
            )
            continue
        if entry.get("event") == "agent_failure":
            parts.append(
                f"<details open><summary><b>agent_failure</b></summary>"
                f"<pre>{escape_html(json.dumps(entry, indent=2))}</pre></details>"
            )
            continue
        if "tool" in entry:
            dup = " (cached duplicate)" if entry.get("duplicate_suppressed") else ""
            result_text = entry.get("result_text") or entry.get("result_preview") or ""
            args = json.dumps(entry.get("arguments", {}), indent=2)
            parts.append(
                f"<details><summary><b>{escape_html(entry.get('source', 'llm'))}</b> "
                f"{escape_html(entry['tool'])}{dup}</summary>"
                f"<p><b>Arguments</b></p><pre>{escape_html(args)}</pre>"
                f"<p><b>Result</b></p><pre>{escape_html(result_text)}</pre></details>"
            )
        else:
            parts.append(f"<pre>{escape_html(json.dumps(entry, indent=2))}</pre>")
    return "".join(parts) or "<i>No audit entries</i>"


def clear_dashboard_display(summary_area, claim_dd, detail_md, audit_acc, status_msg: str) -> None:
    summary_area.value = render_static_summary_html([])
    claim_dd.options = []
    claim_dd.value = None
    detail_md.value = f"<p style='color:#b00020'>{escape_html(status_msg)}</p>"
    audit_acc.children[0].value = "<i>Cleared — no current evaluation snapshot</i>"
    audit_acc.set_title(0, "Tool & validation audit trail")


def build_dashboard():
    import ipywidgets as widgets

    status = widgets.HTML(value="<i>Ready</i>")
    run_btn = widgets.Button(description="Run / Re-run evaluation", button_style="primary")
    claim_dd = widgets.Dropdown(options=[], description="Claim")
    out = widgets.Output()

    summary_area = widgets.HTML()
    detail_md = widgets.HTML()
    audit_acc = widgets.Accordion(children=[widgets.HTML()], selected_index=None)

    def refresh_summary():
        summary_area.value = render_static_summary_html(EVAL_RESULTS)

    def refresh_detail(claim_id: str | None):
        if not claim_id or not EVAL_RESULTS or not EVAL_LAST_RUN_SUCCESS:
            detail_md.value = "<i>No claim selected</i>"
            audit_acc.children[0].value = ""
            return
        r = next(x for x in EVAL_RESULTS if x.claim_id == claim_id)
        pending_note = (
            f"<p><b>Pending review:</b> ${r.pending_amount:,.2f}</p>" if r.pending_amount else ""
        )
        tech = " <em>(technical failure)</em>" if r.technical_failure else ""
        detail_md.value = (
            f"<h4>{escape_html(r.claim_id)} — {escape_html(r.decision)}{tech}</h4>"
            f"<p><b>Approved:</b> ${r.approved_amount:,.2f} &nbsp; <b>Deducted:</b> ${r.deducted_amount:,.2f}</p>"
            f"{pending_note}"
            f"<p><b>Confidence:</b> {r.confidence:.3f}</p>"
            f"<p><b>Policy refs:</b> {escape_html(', '.join(r.policy_refs))}</p>"
            f"<p><b>Missing docs:</b> {escape_html(', '.join(r.missing_docs) or '—')}</p>"
            f"<p><b>Tools used (LLM):</b> {escape_html(', '.join(r.tools_used))}</p>"
            f"<p>{escape_html(r.explanation)}</p>"
        )
        audit_acc.children[0].value = render_audit_html(r.audit_trail)
        audit_acc.set_title(0, "Tool & validation audit trail")

    def on_claim_change(change):
        if change["name"] == "value":
            refresh_detail(change["new"])

    def on_run(_):
        if EVAL_BUSY:
            status.value = "<b style='color:orange'>Evaluation already running</b>"
            return
        run_btn.disabled = True
        status.value = "<b>Running evaluation…</b>"
        with out:
            out.clear_output()
            try:
                run_evaluation_once()
                status.value = (
                    f"<b style='color:green'>Done</b> — model {escape_html(EVAL_MODEL_USED or '')} "
                    f"(run {escape_html(EVAL_RUN_ID or '')})"
                )
                claim_dd.options = [(r.claim_id, r.claim_id) for r in EVAL_RESULTS]
                if EVAL_RESULTS:
                    claim_dd.value = EVAL_RESULTS[0].claim_id
                refresh_summary()
                refresh_detail(claim_dd.value)
            except Exception as exc:  # noqa: BLE001
                msg = str(exc)
                status.value = f"<b style='color:red'>Failed / incomplete</b>"
                clear_dashboard_display(
                    summary_area,
                    claim_dd,
                    detail_md,
                    audit_acc,
                    f"Evaluation failed: {msg}",
                )
                print(msg)
            finally:
                run_btn.disabled = False

    run_btn.on_click(on_run)
    claim_dd.observe(on_claim_change, names="value")

    if EVAL_RESULTS and EVAL_LAST_RUN_SUCCESS:
        status.value = (
            f"<b>Loaded</b> {len(EVAL_RESULTS)} results — run {escape_html(EVAL_RUN_ID or '')}"
        )
        claim_dd.options = [(r.claim_id, r.claim_id) for r in EVAL_RESULTS]
        claim_dd.value = EVAL_RESULTS[0].claim_id
        refresh_summary()
        refresh_detail(claim_dd.value)
    else:
        clear_dashboard_display(summary_area, claim_dd, detail_md, audit_acc, "Awaiting successful evaluation.")

    ui = widgets.VBox(
        [
            widgets.HTML("<h2>Interactive dashboard</h2>"),
            status,
            run_btn,
            summary_area,
            claim_dd,
            detail_md,
            audit_acc,
            out,
        ]
    )
    return ui

## Evaluation (Run All)

This cell runs the agent once for all five claims when credentials are configured. Results populate `EVAL_RESULTS` for the dashboard and JSON export.

In [4]:
# Gate live evaluation before API calls (no mock fallback; .env loaded in setup cell)
EVAL_RESULTS = run_evaluation_once()
display(HTML(render_static_summary_html(EVAL_RESULTS)))
render_sample_outputs(EVAL_RESULTS)

## Sample outputs (snapshot `88f199d28f` @ 2026-11-19 18:50:02 UTC)

_Historical notebook outputs may show older snapshots; only a successful current run is authoritative._

### CLM-001 — APPROVE

<IPython.core.display.JSON object>

### CLM-002 — REJECT

<IPython.core.display.JSON object>

### CLM-003 — PARTIAL_APPROVE

<IPython.core.display.JSON object>

## Dashboard

In [5]:
display(build_dashboard())

## Design Notes & Reasoning

### Architecture
- **OpenAI Chat Completions** with native `tools` / function calling (validated against SDK 3.x `client.chat.completions.create`).
- **Bounded agent loop** (max iterations, duplicate call suppression, argument validation via tool handlers).
- **Deterministic finalize** recomputes money with `Decimal`, enforces Manual Review precedence, and reconciles finalized decisions.

### Amount semantics
| Decision | approved_amount | deducted_amount | pending |
|----------|-----------------|-----------------|---------|
| APPROVE / PARTIAL / REJECT | finalized reimbursable / deductions | per policy | 0 |
| MANUAL_REVIEW | **0** | **0** | full claim total pending |

Provisional per-diem math may appear **only in explanations** for manual-review claims (e.g. CLM-005: $75/$145 meal cap is provisional).

### Policy interpretations
- **Timeliness:** trip-interval bound (see README); advance purchase timing cannot be verified.
- **Aggregates:** explicit `@ $X/day` or `N nights` text is mock evidence; `~$60/day` disclosed as approximate.
- **CLM-004:** retains “3 nights” vs two calendar nights from trip dates — surfaced as discrepancy, not rewritten.
- **Business class:** pending exception review (`POL-AIR-01`); not auto-deducted; director tier treated as **conditional** while airfare unresolved.
- **Receipt flags:** `Yes` satisfies receipt requirement without file inspection.

### Tool design
Tools accept **`claim_id`** (and `rule_ids`) only — amounts are always read from canonical `CLAIMS` data.
`tools_used` lists **successful**, claim-matched tool completions only (failed or mismatched attempts appear in the audit trail).

### Confidence (0–1)
Subjective confidence in **routing / recommendation quality**, not calibrated payment probability.

### Trade-offs & next steps
- Add receipt ingestion / OCR in production; manager workflow for manual review decisions.
- Persist audit trails and human approvals; eval harness with labeled edge cases.
- Optional: `python-dotenv` documented but not required.

### Assumptions & limitations
- Expenses occur within trip dates; no line-item expense dates in dataset.
- Single-day meal lines use one trip day for per-diem (CLM-005), not multiplied by diners.
- Provider failures → `MANUAL_REVIEW` with explicit technical-incomplete messaging (not fabricated policy outcomes).

## Final JSON export

The next cell prints **only** the JSON array required for submission (no extra logging).

In [6]:
import json
if not EVAL_LAST_RUN_SUCCESS or not EVAL_RESULTS or len(EVAL_RESULTS) != 5:
    raise RuntimeError(
        "Evaluation results unavailable or last run failed. Configure OPENAI_API_KEY/.env and Run All."
    )
validate_result_set(EVAL_RESULTS)
assert_live_evaluation_success(EVAL_RESULTS)
print(json.dumps(results_to_export_json(EVAL_RESULTS), indent=2))

[
  {
    "claim_id": "CLM-001",
    "decision": "APPROVE",
    "approved_amount": 1110.0,
    "deducted_amount": 0.0,
    "missing_docs": [],
    "policy_refs": [
      "POL-RCT-01",
      "POL-RCT-02",
      "POL-TIME-01",
      "POL-AIR-01",
      "POL-PD-01",
      "POL-PD-02",
      "POL-CAT-01",
      "POL-APR-02"
    ],
    "confidence": 0.95,
    "explanation": "All submitted receipts are present and eligible. The claim was submitted timely, and all expenses fall within the approved categories. The approval tier is manager, and no exceptions are triggered.",
    "tools_used": [
      "calculate_per_diem",
      "check_airfare_exception",
      "check_approval_tier",
      "check_category_eligibility",
      "check_receipts",
      "check_timeliness"
    ]
  },
  {
    "claim_id": "CLM-002",
    "decision": "REJECT",
    "approved_amount": 0.0,
    "deducted_amount": 380.0,
    "missing_docs": [],
    "policy_refs": [
      "POL-CAT-02",
      "POL-RCT-01",
      "POL-RCT-02",
 